# InsightForge AI — Main Pipeline
### Runs all 5 notebooks in sequence automatically
**Order:** data_loader → eda → charts → insights → report

**How to use:**
1. Paste your Gemini API key in the widget box at the top
2. Change the dataset path widget if using a different CSV
3. Click Run All
4. Wait for all cells to complete
5. Download your PDF report from the final cell


In [ ]:
# These appear as input boxes at the top of the notebook
dbutils.widgets.text(
    "dataset_path",
    "/Volumes/insight/default/titanic/Titanic.csv",
    "Dataset Path"
)
dbutils.widgets.text(
    "gemini_key",
    "",
    "Gemini API Key"
)

dataset_path = dbutils.widgets.get("dataset_path")
gemini_key   = dbutils.widgets.get("gemini_key")

print("✅ Widgets ready")
print(f"   Dataset path : {dataset_path}")
print(f"   API key      : {len(gemini_key)} characters")


In [ ]:
%pip install pandas plotly google-generativeai fpdf2 openpyxl


In [ ]:
dbutils.library.restartPython()


In [ ]:
# Must re-read widgets after restart
dataset_path = dbutils.widgets.get("dataset_path")
gemini_key   = dbutils.widgets.get("gemini_key")

print("✅ Widgets re-loaded after restart")
print(f"   Dataset : {dataset_path}")
print(f"   API key : {len(gemini_key)} characters")


In [ ]:
print("=" * 55)
print("STEP 1 — Loading Data")
print("=" * 55)

import pandas as pd

df = pd.read_csv(dataset_path)

print(f"✅ Data loaded successfully")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print(f"   Columns : {df.columns.tolist()}")


In [ ]:
print("=" * 55)
print("STEP 2 — Running EDA")
print("=" * 55)

def get_basic_summary(df):
    return {
        "rows"                : df.shape[0],
        "columns"             : df.shape[1],
        "column_names"        : list(df.columns),
        "dtypes"              : df.dtypes.astype(str).to_dict(),
        "missing_values"      : df.isnull().sum().to_dict(),
        "missing_percent"     : (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicates"          : int(df.duplicated().sum()),
        "numeric_columns"     : list(df.select_dtypes(include="number").columns),
        "categorical_columns" : list(df.select_dtypes(include="object").columns),
        "total_missing_cells" : int(df.isnull().sum().sum()),
        "memory_usage_kb"     : round(df.memory_usage(deep=True).sum() / 1024, 2),
    }

def get_statistical_summary(df):
    numeric_df = df.select_dtypes(include="number")
    result = {}
    for col in numeric_df.columns:
        result[col] = {
            "mean"    : round(numeric_df[col].mean(),   2),
            "median"  : round(numeric_df[col].median(), 2),
            "std"     : round(numeric_df[col].std(),    2),
            "min"     : round(numeric_df[col].min(),    2),
            "max"     : round(numeric_df[col].max(),    2),
            "skewness": round(numeric_df[col].skew(),   2),
            "kurtosis": round(numeric_df[col].kurt(),   2),
        }
    return result

def get_correlation_matrix(df):
    numeric_df = df.select_dtypes(include="number")
    if numeric_df.shape[1] < 2:
        return None
    return numeric_df.corr().round(2)

def get_value_counts(df, top_n=10):
    cat_cols = df.select_dtypes(include="object").columns
    result   = {}
    for col in cat_cols:
        result[col] = df[col].value_counts().head(top_n).to_dict()
    return result

# Run all EDA functions
summary = get_basic_summary(df)
stats   = get_statistical_summary(df)
corr    = get_correlation_matrix(df)
vc      = get_value_counts(df)

print(f"✅ EDA complete")
print(f"   Numeric columns    : {summary['numeric_columns']}")
print(f"   Categorical columns: {summary['categorical_columns']}")
print(f"   Total missing cells: {summary['total_missing_cells']}")
print(f"   Duplicates         : {summary['duplicates']}")


In [ ]:
print("=" * 55)
print("STEP 3 — Generating Charts")
print("=" * 55)

import plotly.express as px

# Chart 1 — Age Distribution
fig1 = px.histogram(
    df, x="Age",
    title    = "Age Distribution",
    template = "plotly_white",
    color_discrete_sequence=["#636EFA"]
)
fig1.show()

# Chart 2 — Survival Count
survived_counts = df["2urvived"].value_counts().reset_index()
survived_counts.columns = ["Survived", "Count"]
survived_counts["Survived"] = survived_counts["Survived"].map(
    {0: "Did Not Survive", 1: "Survived"}
)
fig2 = px.bar(
    survived_counts, x="Survived", y="Count",
    title    = "Survival Count",
    color    = "Survived",
    template = "plotly_white",
    color_discrete_map={
        "Survived"       : "#00CC96",
        "Did Not Survive": "#EF553B"
    }
)
fig2.show()

# Chart 3 — Correlation Heatmap
corr_matrix = df.select_dtypes("number").corr().round(2)
fig3 = px.imshow(
    corr_matrix,
    text_auto = True,
    title     = "Correlation Heatmap",
    color_continuous_scale = "RdBu_r",
    template  = "plotly_white"
)
fig3.show()

# Chart 4 — Survival by Gender
survival_sex = df.groupby("Sex")["2urvived"].mean().reset_index()
survival_sex.columns = ["Sex", "Survival Rate"]
survival_sex["Survival Rate"] = survival_sex["Survival Rate"].round(2)
fig4 = px.bar(
    survival_sex, x="Sex", y="Survival Rate",
    title    = "Survival Rate by Gender",
    color    = "Sex",
    template = "plotly_white",
    text     = "Survival Rate",
    color_discrete_sequence=["#636EFA","#EF553B"]
)
fig4.update_traces(textposition="outside")
fig4.show()

print(f"✅ Charts complete — 4 charts generated")


In [ ]:
print("=" * 55)
print("STEP 4 — Generating AI Insights")
print("=" * 55)

import google.generativeai as genai

genai.configure(api_key=gemini_key)
model = genai.GenerativeModel("gemini-flash-latest")

def generate_insights(df, summary):
    prompt = f"""
You are a senior data scientist and business analyst.
Analyse this dataset and provide actionable business insights.

Dataset Overview:
- Rows                : {summary["rows"]}
- Columns             : {summary["columns"]}
- Column names        : {summary["column_names"]}
- Numeric columns     : {summary["numeric_columns"]}
- Categorical columns : {summary["categorical_columns"]}
- Missing values      : {summary["missing_values"]}
- Duplicate rows      : {summary["duplicates"]}

Statistical Summary:
{df.describe().to_string()}

Sample Data (first 5 rows):
{df.head(5).to_string()}

Please provide:
1. Dataset Overview      — what kind of data is this?
2. Key Findings          — 5 most important observations
3. Business Insights     — actionable recommendations
4. Data Quality Issues   — problems and how to fix them
5. Suggested Next Steps  — what analysis to do next

Be specific. Use actual numbers from the data.
"""
    return model.generate_content(prompt).text

print("Sending data to Gemini AI — please wait 10 seconds...")
insights = generate_insights(df, summary)

print("✅ AI Insights complete")
print()
print(insights)


In [ ]:
print("=" * 55)
print("STEP 5 — Generating PDF Report")
print("=" * 55)

from fpdf import FPDF
from datetime import datetime

class InsightReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 15)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 12, "InsightForge AI - Data Analysis Report",
                  align="C", fill=True,
                  new_x="LMARGIN", new_y="NEXT")
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 9)
        self.cell(0, 6,
                  f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                  align="C", new_x="LMARGIN", new_y="NEXT")
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10,
                  f"InsightForge AI  |  Page {self.page_no()}",
                  align="C")

    def section_title(self, title):
        self.set_font("Helvetica", "B", 12)
        self.set_fill_color(235, 245, 255)
        self.cell(0, 8, title, fill=True,
                  new_x="LMARGIN", new_y="NEXT")
        self.ln(2)

pdf = InsightReport()
pdf.add_page()

# Section 1 — Overview
pdf.section_title("1.  Dataset Overview")
pdf.set_font("Helvetica", "", 11)
pdf.cell(0, 7, f"  Rows           : {summary['rows']}",          new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 7, f"  Columns        : {summary['columns']}",       new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 7, f"  Duplicates     : {summary['duplicates']}",    new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 7, f"  Missing cells  : {summary['total_missing_cells']}", new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 7, f"  Memory         : {summary['memory_usage_kb']} KB",  new_x="LMARGIN", new_y="NEXT")
pdf.ln(4)

# Section 2 — Missing Values
pdf.section_title("2.  Missing Values")
pdf.set_font("Helvetica", "", 11)
has_missing = False
for col, count in summary["missing_values"].items():
    if count > 0:
        pct = summary["missing_percent"][col]
        pdf.cell(0, 7, f"  {col:20} : {count} missing ({pct}%)", new_x="LMARGIN", new_y="NEXT")
        has_missing = True
if not has_missing:
    pdf.cell(0, 7, "  No missing values — dataset is complete!", new_x="LMARGIN", new_y="NEXT")
pdf.ln(4)

# Section 3 — Stats table
pdf.section_title("3.  Statistical Summary")
stats_df  = df.describe().round(2)
col_count = len(stats_df.columns)
col_w     = 160 / (col_count + 1)
pdf.set_font("Helvetica", "B", 9)
pdf.set_fill_color(200, 220, 255)
pdf.cell(col_w, 6, "Stat", border=1, fill=True)
for col in stats_df.columns:
    pdf.cell(col_w, 6, str(col)[:10], border=1, fill=True)
pdf.ln()
pdf.set_font("Helvetica", "", 8)
for idx in stats_df.index:
    pdf.cell(col_w, 5, str(idx), border=1)
    for col in stats_df.columns:
        pdf.cell(col_w, 5, str(stats_df.loc[idx, col]), border=1)
    pdf.ln()
pdf.ln(6)

# Section 4 — AI Insights
pdf.add_page()
pdf.section_title("4.  AI Generated Insights")
pdf.set_font("Helvetica", "", 10)

# Clean markdown and replace Unicode characters with ASCII equivalents
clean = (insights
         .replace("**", "")
         .replace("##", "")
         .replace("#",  "")
         .replace("*",  "-")
         .replace("\u2013", "-")   # en-dash
         .replace("\u2014", "--")  # em-dash
         .replace("\u2018", "'")   # left single quote
         .replace("\u2019", "'")   # right single quote
         .replace("\u201c", '"')   # left double quote
         .replace("\u201d", '"')   # right double quote
         .replace("\u2026", "...") # ellipsis
         .encode('ascii', 'ignore').decode('ascii'))  # Remove any remaining non-ASCII

pdf.multi_cell(0, 5, clean)

# Save directly to Unity Catalog Volume
output_path = "/Volumes/insight/default/titanic/insightforge_report.pdf"
pdf.output(output_path)

print(f"✅ PDF saved to : {output_path}")
print(f"   Download via: Files → Volumes → insight → default → titanic")


In [ ]:
print()
print("=" * 55)
print("   INSIGHTFORGE AI — PIPELINE COMPLETE ✅")
print("=" * 55)
print()
print("  Step 1 — Data Loader  ✅")
print(f"           {df.shape[0]} rows x {df.shape[1]} columns loaded")
print()
print("  Step 2 — EDA          ✅")
print(f"           {len(summary['numeric_columns'])} numeric + {len(summary['categorical_columns'])} categorical columns analysed")
print()
print("  Step 3 — Charts       ✅")
print(f"           4 charts generated")
print()
print("  Step 4 — AI Insights  ✅")
print(f"           {len(insights)} characters of insights generated")
print()
print("  Step 5 — PDF Report   ✅")
print(f"           Saved to /dbfs/FileStore/insightforge_report.pdf")
print()
print("  To download your PDF:")
print("  Catalog → FileStore → insightforge_report.pdf → Download")
print()
print("=" * 55)
